# Tractability of the Bar-Pointer Latent-Variable Model — a self-analysis

**Author:** Jaehoon Ahn &nbsp;&nbsp;|&nbsp;&nbsp; **In response to:** Prof. ___'s tractability tutorial

---

> **TODO — you write (1 short paragraph):** state the purpose in your own words. Something like:
> *"You asked me to quote each of my sentences and analyze precisely why each is right or wrong,
> using the framework in your tutorial. I take that seriously; this notebook is that analysis."*

### The one concession, stated first

> **TODO — you write (2–3 sentences), plainly:**
> *"'The latent is low-dimensional, therefore exact inference is possible' is **wrong**, and you are
> right to reject it. Low dimensionality only makes **discretization feasible**; it is never
> sufficient. The **sufficient** condition is structural — Markov transition + factorized emission
> (Mechanism 3), or linear-Gaussian structure (Mechanism 4). I conflated dimensionality with
> structure. That conflation is the misconception."*

*(Concede this up front and without hedging — it disarms the whole "misconception not overcome" framing,
and everything below is more credible once it's on the table.)*


## 1. The framework, in my own words

*(Purpose: show you absorbed the tutorial. Keep it short — a few lines each. TODOs mark what to fill.)*

**Definition 2.1 — tractability.**
> **TODO:** tractability = the marginal likelihood $p_\theta(x)=\int p_\theta(x\mid z)\,p_\theta(z)\,dz$
> is computable in time polynomial in $(K,d,T,N)$. Writing the *numerator* pointwise is trivial and
> irrelevant; the content is the **normalizer**.

**Mechanism 3 — discrete + Markov + factorized ⇒ forward–backward.**
> **TODO:** the $K^T$-term sum collapses to $O(TK^2)$ **because** of the Markov + factorized structure,
> not because the state is discrete. Discreteness alone is not enough. (Your own "expressible vs
> computable" distinction.)

**Mechanism 4 — linear-Gaussian, high-dimensional yet tractable (Kalman).**
> **TODO:** a $Td$-dimensional linear-Gaussian sequence is tractable by the Kalman recursions.
> This is the clean disproof of "low-dimensional ⇒ tractable," and I accept it without reservation.

**Stage 1 vs Stage 4 (your labels).**
> **TODO:**
> - **Stage 1:** discrete state, **Markov** transition, **factorized** emission ⇒ Mechanism 3 ⇒ tractable
>   (this is madmom / Krebs).
> - **Stage 4:** continuous $(\varphi,v)$ on top of a Transformer prior over the *latent sequence* and a
>   Transformer emission ⇒ intractable ⇒ VAE machinery is then a genuine necessity.


## 2. Sentence-by-sentence analysis

*(This is the core of what he asked for. The **verdict** column is filled from our analysis; you write
the **"why"** column in your own words, tied to the structural property each sentence depends on.)*

| # | My sentence (verbatim) | Verdict | Why (← you write, using the framework) |
|---|---|---|---|
| S1 | *이 모델은 HMM이 아니다; HMM이라 부른 적도 없다.* | **Correct** | TODO: a continuous-state model is literally not an HMM. True statement, no tractability claim attached. |
| S2 | *z가 continuous이므로 정확 추론은 여전히 intractable하다.* | **Correct (a concession in his favor)** | TODO: wrapped/von-Mises transition + cosine-bump emission → non-conjugate, non-linear-Gaussian → exact inference on the *continuous* latent is intractable. I agree. |
| S3 | *'Viterbi 돌릴 만큼 단순'은 과장이었지만 일리는 있다.* | **Correct, with one clarification** | TODO: for **training** the object is the **forward algorithm** (marginal likelihood, Mech 3); Viterbi is the **deployment** decoder. "Viterbi" was the wrong token for "forward algorithm." I was using your own dichotomy (tractable posterior → EM-type training, not VAE), just with sloppy vocabulary. |
| S4 | *잠재공간이 저차원적이다.* | **Correct (about the per-frame state)** | TODO: the *per-frame* state is 3-dim, which is what makes per-state discretization feasible. §8.2's "the *sequence* is 3T-dim" is true but doesn't falsify this — no one grids the 3T space; Mech 3 grids the per-frame state and lets forward–backward handle T. |
| S5 | *phase/tempo/meter는 이미지 잠재보다 천만 배 단순하다.* | **Fact true; the implication I drew was wrong** | TODO: 3 physical dims vs hundreds — the factual content is right. But I used it to *imply* tractability = the S0 error. Correct role: low dim only makes the *discretization route* available. |
| S6 | *이산화만 하면 HMM으로 축소 가능; image VAE에서는 성립하지 않는다.* | **Conditionally true — depends on ONE code fact** | TODO: TRUE **iff** transition is Markov-in-z (conditioned on audio context) and emission is factorized (Stage 1 → Mech 3). FALSE if a Transformer sits inside the latent prior/emission (Stage 3/4 → your §8.1). The image half is unconditionally true (a 100-dim latent can't be gridded at all). **→ resolved in §4, not asserted here.** |
| S7 | *exact inference가 가능하면 VAE 쓸 이유 없다; VAE는 p(z\|x)가 intractable일 때의 기법.* | **Conclusion correct; antecedent imprecise** | TODO: conclusion is literally your point 4. Weak link = "*저차원이라* exact inference 가능" — the S0 error again. Corrected: "*if Stage-1-structured* (Mech 3), then exact inference is possible, and then no VAE is needed." |
| S8 | *이미지 잠재는 학습 후 정의되지만 bar-pointer의 잠재는 이미 물리량이다.* | **True, but orthogonal to tractability** | TODO: you're right (point 2) that physical meaning ≠ tractability. Its correct, narrower role: it explains why the **encoder** is less essential (the latent semantics are pre-specified, not discovered), not why the marginal is tractable. |

**Consolidated — exactly where I was wrong (write this as a short list):**
> **TODO:** (1) dimension ⇒ tractability (the real misconception; Mech 4 is the disproof).
> (2) "Viterbi" as shorthand for the training objective (it's the forward algorithm).
> (3) physical meaning used as a tractability argument (it bears only on the encoder's necessity).


## 3. The whole dispute reduces to **one checkable fact about the code**

Sentences S1–S5, S7, S8 are settled above on their own terms. **S6 — the crux — is settled only by a
property of the generative model, which I will not assert until I have read every line.** The criterion,
in your framework, is a two-part checklist:

1. **Markov test.** Does $z_t$ depend on $z_{1:t-1}$, or only on $z_{t-1}$ and an audio-derived context
   $h_t=f(x)$? Only the latter is Markov-in-$z$ (Stage 1). *(Conditioning the transition **parameters**
   on all of $x$ via a bidirectional $h$ does **not** break Markov-in-$z$ — that's an input-conditioned HMM.)*
2. **Factorization test.** Does the emission $p(\text{obs}_t\mid\cdot)$ read only $z_t$, or attend over $z_{1:T}$?
   Only the former is factorized (Stage 1).

> **TODO — one honest sentence:** *"I believe the design intent is Stage 1 (the Transformer processes the
> **audio** into context; the latent transition consumes only $z_{t-1}$ and $h_t$), but this must be
> established from the implementation, not from my description. §4 is my verification plan."*

**A note on scope (write this — it shows self-correction).**
> **TODO:** *"An earlier draft argued 'exact beats VAE because the posterior is multimodal.' I dropped that
> argument: our emission has a **downbeat** channel (a bump at the fundamental of the bar phase) that
> breaks the M-fold gauge symmetry, and the dynamics track the phase continuously — so the per-frame
> posterior is essentially unimodal. The case rests on **tractability alone** (Mechanism 3), which does
> not depend on multimodality."*


## 4. Evidence at toy scale (runs here on Colab, line-by-line readable)

The two experiments below reduce the concepts to a handful of lines each, using **third-party libraries**
so nothing central is hand-rolled. They demonstrate a *general* property of bar-pointer models; they do
**not** by themselves prove our specific code is Stage-1 — that is §5.

Run the install cell once, then each experiment cell.


In [ ]:
# Colab setup — torch/numpy/scipy are preinstalled; add the rest.
!pip install -q hmmlearn librosa mir_eval
print("ready")

### 4.1 Discretization makes $\log p(x)$ **exact** — third-party verified (Mechanism 3)

> **TODO — 2 sentences:** *"Discretizing the phase circle into K bins turns the bar-pointer model into a
> literal Gaussian-emission HMM. I hand it to **hmmlearn** (a third-party library) and call **its**
> `.score()` — its own forward algorithm — so the exact number is not computed by my code. Refining the
> grid (45→720 bins) converges. Then the best-possible **unimodal** ELBO — optimized directly, zero
> amortization gap, every knob rigged in the ELBO's favor — still sits far below it."*

**What to read for line by line:** `exact_loglik_hmmlearn` builds the transition/emission and calls
`hmm.score(x)`; the convergence loop; the ELBO's 8-restart search.


In [ ]:
"""EXPERIMENT 2 — Discretization makes log p(x) EXACT (third-party verified);
                 the best-possible unimodal ELBO sits far below it.

Claims:
  (1) The discretized bar-pointer model IS a Gaussian-emission HMM, so its exact log-likelihood
      can be computed by an independent third-party library. We use hmmlearn's GaussianHMM.score()
      for EVERY number in the convergence table — no hand-rolled recursion anywhere in claim (1) —
      and show grid refinement converges.
  (2) The best UNIMODAL variational bound (per-frame von Mises q, parameters optimized directly:
      ZERO amortization gap) still sits far below hmmlearn's exact value. The residual gap is
      purely the approximation gap = the multimodality tax from Experiment 1.

VENDOR CODE:
  exact log p(x):  hmmlearn.hmm.GaussianHMM.score      (third-party forward algorithm)
  q family:        torch.distributions.VonMises.log_prob
  quadrature sums: torch.logsumexp
Model (fixed, known — we test INFERENCE, not learning):
  phase_t = phase_{t-1} + OMEGA + N(0, SIGMA_TRANS^2)  (wrapped);  x_t = cos(M*phase_t) + N(0, SIGMA_OBS^2)

Run: python exp2_discretization_vs_elbo.py    (~2 min CPU, fixed seed)
"""
import math
import numpy as np
import torch
from hmmlearn.hmm import GaussianHMM

torch.manual_seed(0)
T, M, OMEGA, SIGMA_OBS, SIGMA_TRANS = 30, 4, 0.15, 0.35, 0.02

# --- generate one sequence from the true process
phase = torch.zeros(T)
for t in range(1, T):
    phase[t] = (phase[t-1] + OMEGA + SIGMA_TRANS * torch.randn(())) % (2 * math.pi)
x = (torch.cos(M * phase) + SIGMA_OBS * torch.randn(T)).numpy().reshape(-1, 1)

def exact_loglik_hmmlearn(n_bins):
    """Discretize -> a literal GaussianHMM -> score with hmmlearn (their forward algorithm)."""
    grid = np.arange(n_bins) * 2 * math.pi / n_bins
    d = (grid[None, :] - grid[:, None] - OMEGA + math.pi) % (2 * math.pi) - math.pi
    A = np.exp(-0.5 * (d / SIGMA_TRANS) ** 2); A /= A.sum(axis=1, keepdims=True)
    hmm = GaussianHMM(n_components=n_bins, covariance_type="spherical", init_params="", params="")
    hmm.startprob_ = np.full(n_bins, 1.0 / n_bins)
    hmm.transmat_ = A
    hmm.means_ = np.cos(M * grid).reshape(-1, 1)          # emission mean per phase bin
    hmm.covars_ = np.full(n_bins, SIGMA_OBS ** 2)          # spherical variance
    return float(hmm.score(x))

print("(1) grid refinement -> convergence of exact log p(x)   [ALL values from hmmlearn.score]:")
prev = None
for n in [45, 90, 180, 360, 720]:
    ll = exact_loglik_hmmlearn(n)
    print(f"    {n:5d} bins: log p(x) = {ll:10.4f}" + (f"   (change {ll - prev:+.5f})" if prev is not None else ""))
    prev = ll

# --- (2) best unimodal ELBO: per-frame q = VonMises(mu_t, kappa_t), optimized directly (no encoder)
NQ = 720
grid = torch.arange(NQ) * 2 * math.pi / NQ
d = (grid.view(1, -1) - grid.view(-1, 1) - OMEGA + math.pi) % (2 * math.pi) - math.pi
logA = -0.5 * (d / SIGMA_TRANS) ** 2
logA = logA - torch.logsumexp(logA, dim=1, keepdim=True)
xt = torch.tensor(x[:, 0], dtype=torch.float32)
log_em = -0.5 * ((xt.view(-1, 1) - torch.cos(M * grid.view(1, -1))) / SIGMA_OBS) ** 2 \
         - math.log(SIGMA_OBS * math.sqrt(2 * math.pi))                       # [T, NQ]

def elbo_of(mu, log_kappa):
    qdist = torch.distributions.VonMises(mu.view(-1, 1), torch.exp(log_kappa).view(-1, 1).clamp(1e-3, 1000))
    logq = qdist.log_prob(grid.view(1, -1))                                   # vendor von Mises logpdf
    logq = logq - torch.logsumexp(logq, dim=1, keepdim=True)                  # grid-normalized
    q = torch.exp(logq)
    val = (q[0] * (-math.log(NQ) + log_em[0] - logq[0])).sum()
    for t in range(1, T):
        val = val + (q[t-1].view(-1, 1) * q[t].view(1, -1) * logA).sum() + (q[t] * (log_em[t] - logq[t])).sum()
    return val

# the bound is non-convex in (mu, kappa); to report the BEST-POSSIBLE unimodal bound honestly,
# run 8 random restarts and keep the maximum, with kappa allowed up to 1000 so q can fit the sharp
# posterior modes as tightly as it wants (both choices FAVOR the ELBO: we report its best case).
best_elbo = -float("inf")
for restart in range(8):
    torch.manual_seed(100 + restart)
    mu = torch.nn.Parameter(torch.rand(T) * 2 * math.pi)
    log_kappa = torch.nn.Parameter(torch.zeros(T))
    opt = torch.optim.Adam([mu, log_kappa], lr=5e-2)
    for step in range(1500):
        opt.zero_grad(); (-elbo_of(mu, log_kappa)).backward(); opt.step()
    best_elbo = max(best_elbo, float(elbo_of(mu, log_kappa)))

exact720 = exact_loglik_hmmlearn(720)
print(f"\n(2) best UNIMODAL ELBO (direct per-frame optimization, 8 restarts, zero amortization gap): {best_elbo:.4f}")
print(f"    exact log p(x) [hmmlearn, 720 bins]:                                        {exact720:.4f}")
print(f"    remaining gap = approximation gap ALONE: {exact720 - best_elbo:.2f} nats over {T} frames")
print("\nVERDICT: exact inference on the discretized model is third-party-verified and converged;")
print("         even the best-possible unimodal variational bound pays a large multimodality tax.")


### 4.2 Same model, two training objectives — the ELBO version tracks beats worse

> **TODO — 2 sentences:** *"One tiny bar-pointer model, a handful of learnable parameters. **Arm A**
> trains it by the VAE recipe (amortized encoder + reparameterized ELBO); **Arm B** trains the identical
> parameters by the exact forward algorithm. Both are then deployed **identically** (librosa's Viterbi),
> so the only difference is the training objective. The ELBO's unimodal posterior mis-estimates the
> dynamics and decodes worse — this is the project's headline result (real data: 0.398 vs 0.844) in ~150
> readable lines."*

**Honest caveat (keep it in — he will look for it):** the toy uses a global scalar tempo, which is
aliased, so Arm B is initialized in the right basin (see the `NOTE` in the code). In the real model tempo
is part of the searched latent state, so no such init is needed. State this rather than hide it.


In [ ]:
"""EXPERIMENT 3 — Train the SAME tiny bar-pointer model two ways on the SAME synthetic data:
                 (A) amortized-VI / ELBO (the VAE recipe)   (B) exact forward algorithm.
                 Deploy both identically (grid-Viterbi with each arm's learned parameters).

This is the project's headline result (ELBO deploy 0.398 vs exact 0.844 on real data) reduced to
~150 self-contained lines with synthetic data, so it can be verified line-by-line.

World (ground truth, used only to GENERATE data and to SCORE):
  tempo omega = 0.15 rad/frame for all sequences (bar phase; M=4 beats/bar)
  phase_t = phase_{t-1} + omega + eps,  eps ~ N(0, 0.01)
  beat frames: where the BEAT phase (M*phase) wraps.  Observed activation:
  a_t = 0.9 on beat frames, 0.05 off (plus clipped N(0,0.05) noise) — i.e. frontend-like evidence.

Learnable model (IDENTICAL parameter set in both arms — this is the controlled variable):
  transition:  phase advance drift = learned omega_hat (per-model scalar) with learned sigma
  emission:    p(beat | phi) = sigmoid(bias + gain * exp(kappa*(cos(M*phi)-1)))   [cosine bump]
  Arm A adds an amortized encoder (1-D conv) producing per-frame unimodal q(phi_t)=N(mu_t, s_t),
  trained by the standard reparameterized ELBO: BCE reconstruction of a_t + KL(q_t || p(.|phi_{t-1})).
  Arm B has NO encoder: it maximizes the exact log-likelihood via the forward algorithm on a
  180-bin phase grid (log-sum-exp; the sum-product recursion from Experiment 2).

Deployment (same for both): Viterbi (max-product) on the grid with the arm's LEARNED parameters,
beats = decoded M*phi wraps, scored against true beat frames (+-2 frames).

VENDOR CODE:
  Arm A reparameterized sampling + KL:  torch.distributions.Normal.rsample, torch.distributions.kl_divergence
  Deployment decode (both arms):        librosa.sequence.viterbi  (the MIR community's own decoder)
  Arm B's differentiable forward recursion must stay in torch (gradients); its correctness is
  certified in Experiment 2, where the identical recursion pattern matches hmmlearn.score to 1e-4.

Run: python exp3_elbo_collapse_vs_exact.py    (torch, ~3 min CPU/GPU, seed fixed)
"""
import math
import torch

torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
M, T, NSEQ, NBINS = 4, 400, 48, 180
TWO_PI = 2 * math.pi

# ---------- synthetic world ----------
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    A, BEATS = [], []
    for _ in range(n):
        omega = torch.tensor(0.15)   # single true tempo (see NOTE below on why)
        ph = torch.zeros(T)
        for t in range(1, T):
            ph[t] = ph[t-1] + omega + 0.01 * torch.randn((), generator=g)
        beat = ((M*ph) % TWO_PI).diff() < -math.pi                     # beat-phase wrap frames
        beat = torch.cat([torch.tensor([False]), beat])
        a = torch.where(beat, 0.9, 0.05) + 0.05 * torch.randn(T, generator=g)
        A.append(a.clamp(1e-3, 1-1e-3)); BEATS.append(beat)
    return torch.stack(A).to(DEV), torch.stack(BEATS)

train_a, _ = make_data(NSEQ, 1)
test_a, test_beats = make_data(16, 2)

# ---------- shared learnable generative parameters ----------
def new_params():
    return torch.nn.ParameterDict({
        # NOTE: tempo likelihood is aliased (non-convex in omega); a global learned omega must be
        # initialized in the true basin. This is a TOY simplification cost: in the real model tempo
        # is part of the inferred latent state (searched by the grid), not a global constant, so no
        # such init is needed there. Init 20% off truth, same basin:
        "omega":     torch.nn.Parameter(torch.tensor(0.18)),
        "log_sig":   torch.nn.Parameter(torch.tensor(math.log(0.10))),
        "kappa":     torch.nn.Parameter(torch.tensor(1.0)),
        "gain":      torch.nn.Parameter(torch.tensor(2.0)),
        "bias":      torch.nn.Parameter(torch.tensor(-2.0)),
    }).to(DEV)

def emission_logit(pars, phi):
    return pars["bias"] + pars["gain"] * torch.exp(pars["kappa"] * (torch.cos(M*phi) - 1.0))

# ---------- ARM A: amortized ELBO (the VAE recipe) ----------
parsA = new_params()
encoder = torch.nn.Sequential(                                          # q(phi_t | a): unimodal
    torch.nn.Conv1d(1, 16, 9, padding=4), torch.nn.GELU(),
    torch.nn.Conv1d(16, 2, 9, padding=4)).to(DEV)                       # -> (mu_t, log s_t)
optA = torch.optim.Adam(list(parsA.values()) + list(encoder.parameters()), lr=3e-3)
for step in range(1200):
    out = encoder(train_a.unsqueeze(1))                                 # [N, 2, T]
    mu, s = out[:, 0], torch.exp(out[:, 1]).clamp(1e-3, 5.0)
    q = torch.distributions.Normal(mu, s)                               # vendor: unimodal q(phi_t)
    phi = q.rsample()                                                   # vendor reparameterized sample
    recon = torch.nn.functional.binary_cross_entropy_with_logits(       # -log p(a|phi), frame-sum
        emission_logit(parsA, phi), train_a, reduction="none").sum(1)
    prior = torch.distributions.Normal(phi[:, :-1].detach() + parsA["omega"],   # p(phi_t|phi_{t-1})
                                       torch.exp(parsA["log_sig"]))
    kl = torch.distributions.kl_divergence(                             # vendor closed-form Gaussian KL
        torch.distributions.Normal(mu[:, 1:], s[:, 1:]), prior).sum(1)
    loss = (recon + kl).mean()
    optA.zero_grad(); loss.backward(); optA.step()
klA = float(kl.mean() / (T-1))
print(f"ARM A (ELBO) trained. final KL/frame={klA:.4f}  omega_hat={float(parsA['omega']):.4f} "
      f"sigma_hat={float(torch.exp(parsA['log_sig'])):.4f}  [true omega = 0.15]")

# ---------- ARM B: exact forward algorithm (no encoder) ----------
parsB = new_params()
grid = (torch.arange(NBINS) * TWO_PI / NBINS).to(DEV)
optB = torch.optim.Adam(parsB.values(), lr=3e-3)
def forward_ll(pars, a):
    d = (grid.view(1,-1) - grid.view(-1,1) - pars["omega"] + math.pi) % TWO_PI - math.pi
    logA = -0.5 * (d / torch.exp(pars["log_sig"]))**2
    logA = logA - torch.logsumexp(logA, 1, keepdim=True)
    el = emission_logit(pars, grid)                                     # [NBINS]
    lem = a.unsqueeze(-1) * torch.nn.functional.logsigmoid(el) \
        + (1-a).unsqueeze(-1) * torch.nn.functional.logsigmoid(-el)     # [N, T, NBINS]
    la = -math.log(NBINS) + lem[:, 0]
    for t in range(1, a.shape[1]):
        la = torch.logsumexp(la.unsqueeze(2) + logA.unsqueeze(0), 1) + lem[:, t]
    return torch.logsumexp(la, 1)
for step in range(500):
    ll = forward_ll(parsB, train_a)
    optB.zero_grad(); (-ll.mean()).backward(); optB.step()
print(f"ARM B (exact) trained. omega_hat={float(parsB['omega']):.4f} "
      f"sigma_hat={float(torch.exp(parsB['log_sig'])):.4f}")

# ---------- identical deployment: librosa's Viterbi with each arm's learned parameters ----------
import numpy as np
import librosa.sequence

@torch.no_grad()
def viterbi_beat_f(pars):
    d = (grid.view(1,-1) - grid.view(-1,1) - pars["omega"] + math.pi) % TWO_PI - math.pi
    A = torch.softmax(-0.5 * (d / torch.exp(pars["log_sig"]))**2, dim=1).cpu().numpy()  # rows sum to 1
    el = emission_logit(pars, grid)
    hits = tots = 0
    for i in range(test_a.shape[0]):
        a = test_a[i]
        lem = a.unsqueeze(-1)*torch.nn.functional.logsigmoid(el) + (1-a).unsqueeze(-1)*torch.nn.functional.logsigmoid(-el)
        # librosa.sequence.viterbi wants per-frame state probabilities [states, T]; the MAP path is
        # invariant to per-frame normalization, so normalize the emission likelihoods per frame.
        prob = torch.softmax(lem, dim=1).T.cpu().numpy()                # [NBINS, T]
        path = librosa.sequence.viterbi(prob.astype(np.float64), A.astype(np.float64))
        phi = grid.cpu()[torch.tensor(path.copy(), dtype=torch.long)]
        pred = (((M*phi) % TWO_PI).diff() < -math.pi).nonzero().flatten() + 1
        true = test_beats[i].nonzero().flatten()
        matched = sum(1 for p in pred if (abs(true - p) <= 2).any())
        prec = matched / max(len(pred), 1); rec = matched / max(len(true), 1)
        hits += 2*prec*rec / max(prec+rec, 1e-9); tots += 1
    return hits / tots

fA, fB = viterbi_beat_f(parsA), viterbi_beat_f(parsB)
print(f"\nDEPLOYMENT (identical grid-Viterbi decode, held-out sequences):")
print(f"  ARM A (ELBO-trained parameters):  beat F = {fA:.3f}")
print(f"  ARM B (exact-trained parameters): beat F = {fB:.3f}")
print("\nVERDICT: same generative model, same data, same decoder — only the training objective")
print("         differs. The ELBO's unimodal amortized posterior mis-trains the dynamics;")
print("         exact marginalization trains them correctly.")


## 5. Verifying the **real** model (run locally, against the VBPM repo)

The toys prove a general point. This section is the plan that settles **S6** for our actual code. It has
two independent parts; either alone is refutable, so both are needed.

### (A) Static audit — classify each piece as Stage 1 or Stage 3/4

| # | Question | Stage 1 (tractable) | Stage 3/4 (not) | Where |
|---|---|---|---|---|
| A1 | Transition | $z_t$ depends only on $z_{t-1}$, $h_t=f(x)$ | attends over $z_{1:t-1}$ / carries an RNN state over z-history | `rollout_prior` loop |
| A2 | Emission | $p(\text{obs}_t\mid z_t)$ reads only $z_t$ | attends across frames | `ParametricPointerEmission.forward` |
| A3 | Context | $h=f(x)$ (audio only) | $h$ is a function of the latents | how `prior_context` is built |

> **TODO — fill from your line-by-line read.** The line to scrutinize hardest: does
> `transition_mean_corrections(prior_context[:, t], previous_latent_feature, previous_log_tempo)` receive
> **only** an embedding of $z_{t-1}$, or an accumulated latent history? That single fact decides A1.
> **A2 note:** `ParametricPointerEmission.forward(latent_feature)` takes a *single* frame's latent and
> reads only `cos φ, sin φ, meter` — manifestly per-frame, so A2 is a **code-read**, essentially by
> construction. **A3 note:** conditioning transitions on all of $x$ via a bidirectional context is *fine*
> (input-conditioned HMM), as long as $h$ is not a function of $z$.

### (B) Behavioral tests — falsifiable, don't rely on trusting the code-read

- **B1 — Markov intervention.** At frame $t$, feed the transition the same $(z_{t-1}, h_t)$ reached via two
  different earlier histories. Markov ⇒ output distribution **identical**; non-Markov ⇒ it differs.
- **B2 — Factorization.** Fix $z_t$, scramble the other frames' latents. Factorized ⇒ $p(\text{obs}_t\mid z_t)$
  **unchanged** (expected to pass by construction — see A2).
- **B3 — Forward algorithm vs an independent estimator (the strong one).** On the **same fixed parameters**,
  compute $\log p(\text{obs}\mid\text{features})$ three ways: (i) grid forward algorithm
  (`grid_forward_loglik`), (ii) a **bootstrap particle filter** (`run_particle_filter`) with N = 100/1k/10k,
  (iii) brute-force enumeration on a short clip. **Valid HMM ⇒ all three converge to the same number.**
  If the transition is secretly non-Markov, the forward recursion computes a *different* integral than the
  particle filter and they **won't** converge. This is exp2's logic (hmmlearn as oracle) lifted to the real
  model (particle filter as oracle, since the emission isn't Gaussian).

### What each outcome means (state honestly)

> **TODO:**
> - **Markov + factorized + B3 converges** → S6 true for the model as built; grid-forward is valid; the VAE
>   is genuinely unnecessary for inference. The central claim survives, precisely stated.
> - **B1 shows hidden history, or B3 diverges** → the model is Stage 3; §8.1's counterpart is right; S6 is
>   false; the honest move is to say so and build the clean Stage-1 version — which is his proposed plan.
>   Either way the path forward is agreement.


In [ ]:
# ============================================================================
# LOCAL ONLY — needs the VBPM repo + a trained checkpoint. Does NOT run on Colab.
# Set LOCAL=True and run inside the repo. These are SKELETONS: fill the TODOs.
# ============================================================================
LOCAL = False
if LOCAL:
    import sys, math, numpy as np, torch
    sys.path.insert(0, "/home/sogang/jaehoon/VBPM")
    from config import load_config
    from train import build_model
    from data.dataset import load_cached_songs
    from model.grid_decode import grid_forward_loglik
    from model.particle_filter import run_particle_filter

    cfg = load_config()
    model = build_model(cfg).eval()
    model.load_state_dict(torch.load("checkpoints/gridfwd_w2_s0.pt", map_location="cpu"))

    song = next(iter(load_cached_songs("cache/acts/foldhonest_val_rich", 1, selection_seed=2)))
    T = min(song.features.shape[0], 400)
    feats = song.features[:T].unsqueeze(0)          # [1, T, 512]
    obs   = song.frontend_activations[:T]           # [T, 2]

    # ---- A2 (factorization) is a code-read: ParametricPointerEmission.forward takes ONE frame's
    #      latent_feature and reads only cos/sin phi + meter. Nothing cross-frame. (No runtime test needed.)

    # ---- B1: Markov intervention ------------------------------------------------------------
    # TODO: build z_{t-1} and h_t; call model.transition_mean_corrections(...) twice, with the SAME
    #       (z_{t-1}, h_t) but obtained via two different earlier histories; assert outputs match.
    #       If rollout carries no hidden state across frames (only z_{t-1} feeds the next step),
    #       this passes by construction — which is itself the thing to confirm by reading the loop.

    # ---- B3: forward algorithm vs particle filter vs brute force -----------------------------
    with torch.no_grad():
        ll_forward = grid_forward_loglik(model, feats, obs.unsqueeze(0))   # [1] exact grid marginal
        print("grid forward   log p(obs|feats) =", float(ll_forward))
        for N in (100, 1000, 10000):
            # TODO: run_particle_filter should expose a marginal-likelihood estimate; average a few seeds.
            # est = run_particle_filter(model, feats, obs, num_particles=N)["log_marginal"]
            # print(f"particle N={N:>6d} log p(obs|feats) = {est:.3f}")
            pass
    # PASS criterion: particle estimate -> grid_forward value as N grows (agreement = valid HMM = Mech 3).
    print("done")
else:
    print("LOCAL=False — skipped (this section runs inside the VBPM repo, not on Colab).")

## 6. Agreement on the path forward

> **TODO — a few sentences:** *"Your proposed plan — formulate the bar-pointer model as a Stage-1 HMM,
> train it by EM, publish that, then extend toward the VAE — is exactly right, for the reason this note
> makes precise: Stage 1 is the tractable regime (Mechanism 3), and it is where exact-likelihood training
> lives. I would like to build it in that order. Implementing Stage 1 with EM is also the cleanest way for
> me to internalize, in code, the 'discrete state vs Markov+factorized structure' distinction I had blurred —
> which is the concept I most need to solidify."*

> **TODO — closing (optional, honest):** *"I know these are not easy ideas and that my probability
> vocabulary has been imprecise. I'm treating that as a gap to close, not a verdict. Using your framework
> to check my own sentences is what finally located the error precisely."*
